In [42]:
import numpy as np

In [43]:

states = ['E', '5', 'I']
transition_matrix = {
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0, '_': 1.0},
    '5': {'E': 0.0, '5': 0.0, 'I': 1.0, '_': 1.0},
    'I': {'E': 0.0, '5': 0.0, 'I': 0.9, '_': 0.1}
}
start = {'E': 1.0, '5': 0.0, 'I': 0.0, '_': 0.0}
emission_matrix = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

#"_" is a special state that indicates the end of the sequence

In [44]:
def log_prob_of_a_given_path(underlying, sequence):
    if len(underlying) != len(sequence):
        raise ValueError("Length of underlying state sequence and observed sequence must be equal")
    else :
        underlying = underlying + '_'
        prob = 1.0
        for i in range(len(sequence)):
            prob *= transition_matrix[underlying[i]][underlying[i+1]] * emission_matrix[underlying[i]][sequence[i]]
        return round(np.log(prob), 2)
    
log_prob_of_a_given_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA")

-41.22

In [45]:
def viterbi_algorithm(sequence):
    T = len(sequence)
    # V[t][s] = max log-prob up to position t, ending in state s
    V = [ { } for _ in range(T) ]
    # path[s] = best state sequence up to current t ending in s
    path = { s: [s] for s in states }

    for s in states:
        sp = start.get(s, 0.0)
        ep = emission_matrix[s].get(sequence[0], 0.0)
        if sp > 0 and ep > 0:
            V[0][s] = np.log(sp) + np.log(ep)
        else:
            V[0][s] = -np.inf

    for t in range(1, T):
        new_path = {}
        obs = sequence[t]
        for curr in states:
            ep = emission_matrix[curr].get(obs, 0.0)
            if ep == 0:
                # impossible to emit this base in curr
                V[t][curr] = -np.inf
                new_path[curr] = [curr]
                continue

            best_lp = -np.inf
            best_prev = None
            for prev in states:
                tp = transition_matrix[prev].get(curr, 0.0)
                if tp == 0:
                    continue
                lp = V[t-1][prev] + np.log(tp)
                if lp > best_lp:
                    best_lp = lp
                    best_prev = prev

            if best_prev is None:
                V[t][curr] = -np.inf
                new_path[curr] = [curr]
            else:
                V[t][curr] = best_lp + np.log(ep)
                new_path[curr] = path[best_prev] + [curr]

        path = new_path

    best_lp = -np.inf
    best_prev = None
    t = T-1
    for prev in states:
        tp = transition_matrix[prev].get('_', 0.0)
        if tp == 0:
            continue
        lp = V[t][prev] + np.log(tp)
        if lp > best_lp:
            best_lp = lp
            best_prev = prev

    best_path = path[best_prev] + ['_']
    return best_path, best_lp


In [46]:

test_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
most_likely_path, log_prob = viterbi_algorithm(test_sequence)

print(f"Most likely path: {''.join(most_likely_path[:-1])}")
print(f"Log probability: {log_prob:.2f}")


Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.68


In [47]:

test_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
v_path, v_logp = viterbi_algorithm(test_sequence)

#Drop the final '_' so path length == sequence length
aligned_path = v_path[:-1]   

helper_logp = log_prob_of_a_given_path("".join(aligned_path), test_sequence)

print(f"Viterbi returned path (aligned): {''.join(aligned_path)}")
print(f" Viterbi log‐prob:       {v_logp:.2f}")
print(f" Helper log‐prob:        {helper_logp:.2f}")


Viterbi returned path (aligned): EEEEEEEEEEEEEEEEEEEEEEEEEE
 Viterbi log‐prob:       -38.68
 Helper log‐prob:        -38.68


In [48]:

proposed_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
if len(proposed_path) == len(test_sequence):
    proposed_log_prob = log_prob_of_a_given_path(proposed_path, test_sequence)
    print(f"Log probability of the provided path: {proposed_log_prob:.2f}")
    print(f"Log probability of the Viterbi path: {log_prob:.2f}")   
    print(f"Is Viterbi path more likely? {log_prob > proposed_log_prob}")
else:
    print("The provided path length doesn't match the sequence length.")


Log probability of the provided path: -41.22
Log probability of the Viterbi path: -38.68
Is Viterbi path more likely? True
